In [1]:
import pandas as pd
import numpy as np

# DATA CLEANING

In [ ]:
# 1. Đọc dữ liệu thô
file_path = '../datas/raw/binh_duong_real_estates.csv'
try:
    df = pd.read_csv(file_path)
    initial_rows, initial_cols = df.shape
    print(f"=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: {initial_rows:,} dòng | {initial_cols} cột) ===\n")
except FileNotFoundError:
    print(f"[ERROR] Không tìm thấy file dữ liệu thô tại: {file_path}")
    exit()

# 2. [R1] Loại bỏ cột khuyết nặng > 80%
drop_cols = ['house_depth', 'floor_count', 'road_width', 'balcony_direction']
existing_drop_cols = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=existing_drop_cols)
print(f"[R1] Đã loại bỏ {len(existing_drop_cols)} cột khuyết nặng > 80%.")

# 3. [R2 & R5] Xóa dòng khuyết price và area < 10m²
df_clean = df_clean.dropna(subset=['price'])
df_clean = df_clean[df_clean['area'] >= 10]
print(f"[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².")

# 4. [R3] Gán giá trị mặc định cho thuộc tính Dimension văn bản bị khuyết
impute_text_cols = {
    'ward_name': 'Chưa xác định',
    'house_direction': 'Chưa xác định',
    'street_name': 'Chưa xác định',
    'project_name': 'Không thuộc dự án'
}
for col, val in impute_text_cols.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(val)
print(f"[R3] Đã gán giá trị mặc định cho thuộc tính chiều địa lý và dự án khuyết (ward, direction, street, project).")

# 5. [R4] Khử trùng lặp nghiệp vụ (Business Deduplication)
biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
valid_biz_cols = [c for c in biz_cols if c in df_clean.columns]
df_clean = df_clean.drop_duplicates(subset=valid_biz_cols, keep='first')
print(f"[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.")

# 6. [R6] Chuẩn hóa khoảng trắng chuỗi văn bản (Trim/Strip)
text_cols = ['district_name', 'ward_name', 'street_name', 'project_name', 'name']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()
print(f"[R6] Đã xóa khoảng trắng thừa ở đầu/cuối tất cả các chuỗi văn bản.")

# 7. [R7] Xử lý dữ liệu ngoại lệ tổng giá cực đoan (> 500 tỷ VNĐ)
df_outliers = df_clean[df_clean['price'] > 500_000_000_000]
df_clean = df_clean[df_clean['price'] <= 500_000_000_000]
print(f"[R7] Đã lọc bỏ {len(df_outliers)} dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.")

# 8. [R8] Gán giá trị mặc định thuộc tính số kỹ thuật & Ép kiểu an toàn frontage_width
impute_num_cols = {'bedroom_count': 0, 'bathroom_count': 0}
for col, val in impute_num_cols.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(val).astype(int)

if 'frontage_width' in df_clean.columns:
    df_clean['frontage_width'] = pd.to_numeric(df_clean['frontage_width'], errors='coerce')
print(f"[R8] Đã gán số 0 cho số phòng ngủ/WC khuyết và ép kiểu Float chuẩn cho frontage_width.")

# 9. [R9] Tự động nhận dạng đa định dạng ngày tháng và phân tách hạt nhân thời gian
if 'published_at' in df_clean.columns:
    # Sử dụng format='mixed' và dayfirst=True để Pandas tự động đoán định dạng chuẩn xác
    df_clean['published_at_dt'] = pd.to_datetime(
        df_clean['published_at'], 
        format='mixed', 
        dayfirst=True, 
        errors='coerce'
    )
    
    # Ép về định dạng YYYY-MM-DD chuẩn cho Staging SQL
    df_clean['full_date'] = df_clean['published_at_dt'].dt.strftime('%Y-%m-%d')
    
    # Đối với những dòng hoàn toàn thiếu/lỗi không thể đọc được, điền ngày mặc định '1900-01-01'
    df_clean['full_date'] = df_clean['full_date'].fillna('1900-01-01')
    
    # Trích xuất khung giờ (khuyết gán = 0)
    df_clean['post_hour'] = df_clean['published_at_dt'].dt.hour.fillna(0).astype(int)
    
    df_clean = df_clean.drop(columns=['published_at', 'published_at_dt'])
    print(f"[R9] Đã phân tách 'published_at' thành 'full_date' và 'post_hour' (Nhận dạng tự động đa định dạng Datetime).")

# 10. [R10] Gắn nhãn Outliers quy mô bất động sản (is_outlier)
df_clean['is_outlier'] = 0
outlier_condition = (
    (df_clean['bedroom_count'] > 10) | 
    (df_clean['bathroom_count'] > 10) | 
    (df_clean['area'] > 2000)
)
df_clean.loc[outlier_condition, 'is_outlier'] = 1
outlier_count = df_clean['is_outlier'].sum()
print(f"[R10] Đã gắn nhãn 'is_outlier = 1' cho {outlier_count:,} bản ghi có quy mô cực đoan.")

# 11. Xuất file dữ liệu sạch và ngoại lệ
final_rows, final_cols = df_clean.shape
output_clean_path = '../datas/processed/cleaned_binh_duong_real_estates.csv'
output_outlier_path = '../datas/processed/outliers_binh_duong_real_estates.csv'

df_clean.to_csv(output_clean_path, index=False, encoding='utf-8-sig')
df_outliers.to_csv(output_outlier_path, index=False, encoding='utf-8-sig')

print(f"\n=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH (R1 - R10) ===")
print(f"- Số dòng sạch còn lại: {final_rows:,} dòng (Đã lọc bỏ {initial_rows - final_rows:,} dòng rác/ngoại lệ)")
print(f"- Số cột dữ liệu sạch: {final_cols} cột")
print(f"- File dữ liệu sạch đã lưu tại: {output_clean_path}")
print(f"- File dữ liệu ngoại lệ (> 500 tỷ) đã lưu tại: {output_outlier_path}")

=== BẮT ĐẦU TIẾN TRÌNH LÀM SẠCH DỮ LIỆU (Ban đầu: 42,783 dòng | 19 cột) ===

[R1] Đã loại bỏ 4 cột khuyết nặng > 80%.
[R2 & R5] Đã xóa toàn bộ dòng khuyết 'price' và các dòng có 'area' < 10 m².
[R3] Đã gán giá trị mặc định cho thuộc tính chiều địa lý và dự án khuyết (ward, direction, street, project).
[R4] Đã lọc trùng lặp nghiệp vụ dựa trên bộ Khóa nghiệp vụ chuẩn.
[R6] Đã xóa khoảng trắng thừa ở đầu/cuối tất cả các chuỗi văn bản.
[R7] Đã lọc bỏ 17 dòng có tổng giá cực đoan (> 500 tỷ VNĐ) sang vùng lưu trữ riêng.
[R8] Đã gán số 0 cho số phòng ngủ/WC khuyết và ép kiểu Float chuẩn cho frontage_width.
[R9] Đã phân tách 'published_at' thành 'full_date' và 'post_hour' (Nhận dạng tự động đa định dạng Datetime).
[R10] Đã gắn nhãn 'is_outlier = 1' cho 681 bản ghi có quy mô cực đoan.

=== HOÀN THÀNH TIẾN TRÌNH LÀM SẠCH (R1 - R10) ===
- Số dòng sạch còn lại: 36,303 dòng (Đã lọc bỏ 6,480 dòng rác/ngoại lệ)
- Số cột dữ liệu sạch: 17 cột
- File dữ liệu sạch đã lưu tại: ../datas/processed/cleaned_b

In [9]:
import pandas as pd

# 1. Khởi tạo đường dẫn file sạch
clean_path = '../datas/processed/cleaned_binh_duong_real_estates.csv'
outlier_path = '../datas/processed/outliers_binh_duong_real_estates.csv'

print("=" * 70)
print("BÁO CÁO NGHIỆM THU ĐỐI SOÁT DỮ LIỆU SẠCH (POST-ETL CHECK)")
print("=" * 70)

try:
    df_clean = pd.read_csv(clean_path)
    print(f"[OK] Đã tải file sạch thành công: {df_clean.shape[0]:,} dòng | {df_clean.shape[1]} cột\n")
except FileNotFoundError:
    print(f"[ERROR] Không tìm thấy file tại đường dẫn: {clean_path}")
    exit()

# 2. Kiểm tra R1: Cấu trúc cột (Loại bỏ 4 cột khuyết > 80%)
dropped_cols = ['house_depth', 'floor_count', 'road_width', 'balcony_direction']
r1_check = any(col in df_clean.columns for col in dropped_cols)
print(f"1.  [R1] Loại bỏ 4 cột khuyết > 80%: {'THẤT BẠI' if r1_check else 'ĐẠT (Đã loại bỏ trọn vẹn 4 cột)'}")

# 3. Kiểm tra R2: Khuyết giá bất động sản (price)
r2_null_price = df_clean['price'].isnull().sum()
print(f"2.  [R2] Kiểm tra dòng khuyết 'price': {'ĐẠT (0 dòng NULL)' if r2_null_price == 0 else f'THẤT BẠI (Còn {r2_null_price} dòng NULL)'}")

# 4. Kiểm tra R3: Gán giá trị mặc định thuộc tính chiều văn bản
impute_text_cols = {
    'ward_name': 'Chưa xác định',
    'house_direction': 'Chưa xác định',
    'street_name': 'Không xác định',
    'project_name': 'Không thuộc dự án'
}
r3_fails = 0
for col, val in impute_text_cols.items():
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        r3_fails += 1
print(f"3.  [R3] Kiểm tra gán mặc định thuộc tính chiều văn bản: {'ĐẠT (0 dòng NULL)' if r3_fails == 0 else f'THẤT BẠI (Còn {r3_fails} cột bị NULL)'}")

# 5. Kiểm tra R4: Trùng lặp nghiệp vụ (Business Duplicates)
biz_cols = ['name', 'price', 'area', 'ward_name', 'district_name']
valid_biz = [c for c in biz_cols if c in df_clean.columns]
r4_duplicates = df_clean.duplicated(subset=valid_biz).sum()
print(f"4.  [R4] Trùng lặp nghiệp vụ theo Business Key: {'ĐẠT (0 dòng trùng)' if r4_duplicates == 0 else f'THẤT BẠI (Còn {r4_duplicates} dòng trùng)'}")

# 6. Kiểm tra R5: Ranh giới diện tích (area < 10m2)
r5_invalid_area = (df_clean['area'] < 10).sum()
print(f"5.  [R5] Kiểm tra ranh giới 'area' < 10m²: {'ĐẠT (0 dòng vi phạm)' if r5_invalid_area == 0 else f'THẤT BẠI (Còn {r5_invalid_area} dòng area < 10)'}")

# 7. Kiểm tra R6: Khoảng trắng thừa (Trim/Strip)
text_cols = ['district_name', 'ward_name', 'street_name', 'project_name', 'name']
r6_spaces = 0
for col in text_cols:
    if col in df_clean.columns:
        r6_spaces += (df_clean[col].astype(str) != df_clean[col].astype(str).str.strip()).sum()
print(f"6.  [R6] Kiểm tra khoảng trắng thừa văn bản: {'ĐẠT (0 dòng dính khoảng trắng)' if r6_spaces == 0 else f'THẤT BẠI (Còn {r6_spaces} dòng chưa strip)'}")

# 8. Kiểm tra R7: Ngoại lệ tổng giá cực đoan (> 500 tỷ VNĐ)
r7_extreme_price = (df_clean['price'] > 500_000_000_000).sum()
print(f"7.  [R7] Kiểm tra tổng giá cực đoan > 500 tỷ VNĐ: {'ĐẠT (0 dòng vi phạm)' if r7_extreme_price == 0 else f'THẤT BẠI (Còn {r7_extreme_price} dòng > 500 tỷ)'}")

# 9. Kiểm tra R8: Gán giá trị mặc định thuộc tính số kỹ thuật & ép kiểu frontage_width
r8_null_rooms = df_clean[['bedroom_count', 'bathroom_count']].isnull().sum().sum()
r8_frontage_type = pd.api.types.is_numeric_dtype(df_clean['frontage_width'])
print(f"8.  [R8] Kiểm tra thuộc tính số (phòng ngủ/WC/mặt tiền): {'ĐẠT (0 dòng NULL phòng, đúng kiểu Float)' if (r8_null_rooms == 0 and r8_frontage_type) else 'THẤT BẠI'}")

# 10. Kiểm tra R9: Tách cột thời gian (published_at -> full_date & post_hour)
r9_has_dates = 'full_date' in df_clean.columns and 'post_hour' in df_clean.columns
r9_no_raw_date = 'published_at' not in df_clean.columns
print(f"9.  [R9] Kiểm tra phân tách 'full_date' và 'post_hour': {'ĐẠT (Đã tách chuẩn, xóa cột gốc)' if (r9_has_dates and r9_no_raw_date) else 'THẤT BẠI'}")

# 11. Kiểm tra R10: Gắn nhãn Outlier (is_outlier)
r10_has_outlier_col = 'is_outlier' in df_clean.columns
r10_outlier_count = df_clean['is_outlier'].sum() if r10_has_outlier_col else 0
print(f"10. [R10] Kiểm tra cờ gán nhãn 'is_outlier': {'ĐẠT' if r10_has_outlier_col else 'THẤT BẠI'} | Số bản ghi Outlier: {r10_outlier_count:,}")

# 12. Kiểm tra file Staging cô lập Outliers cực đoan (> 500 tỷ)
try:
    df_outliers = pd.read_csv(outlier_path)
    print(f"\n[STAGING CHECK] File cô lập ngoại lệ cực đoan ({outlier_path}): Đã lưu trữ an toàn {len(df_outliers)} dòng.")
except FileNotFoundError:
    print(f"\n[STAGING CHECK] Không tìm thấy file ngoại lệ tại: {outlier_path}")

print("=" * 70)
print("KẾT LUẬN: Tập dữ liệu sạch đạt chuẩn 100% (R1-R10) sẵn sàng đưa vào SSIS / SQL Server.")
print("=" * 70)

BÁO CÁO NGHIỆM THU ĐỐI SOÁT DỮ LIỆU SẠCH (POST-ETL CHECK)
[OK] Đã tải file sạch thành công: 36,303 dòng | 17 cột

1.  [R1] Loại bỏ 4 cột khuyết > 80%: ĐẠT (Đã loại bỏ trọn vẹn 4 cột)
2.  [R2] Kiểm tra dòng khuyết 'price': ĐẠT (0 dòng NULL)
3.  [R3] Kiểm tra gán mặc định thuộc tính chiều văn bản: ĐẠT (0 dòng NULL)
4.  [R4] Trùng lặp nghiệp vụ theo Business Key: ĐẠT (0 dòng trùng)
5.  [R5] Kiểm tra ranh giới 'area' < 10m²: ĐẠT (0 dòng vi phạm)
6.  [R6] Kiểm tra khoảng trắng thừa văn bản: ĐẠT (0 dòng dính khoảng trắng)
7.  [R7] Kiểm tra tổng giá cực đoan > 500 tỷ VNĐ: ĐẠT (0 dòng vi phạm)
8.  [R8] Kiểm tra thuộc tính số (phòng ngủ/WC/mặt tiền): ĐẠT (0 dòng NULL phòng, đúng kiểu Float)
9.  [R9] Kiểm tra phân tách 'full_date' và 'post_hour': ĐẠT (Đã tách chuẩn, xóa cột gốc)
10. [R10] Kiểm tra cờ gán nhãn 'is_outlier': ĐẠT | Số bản ghi Outlier: 681

[STAGING CHECK] File cô lập ngoại lệ cực đoan (../datas/processed/outliers_binh_duong_real_estates.csv): Đã lưu trữ an toàn 17 dòng.
KẾT LUẬN: T

In [1]:
import pandas as pd
df = pd.read_csv('../datas/processed/cleaned_binh_duong_real_estates.csv')
df

,name,description,property_type_name,province_name,district_name,ward_name,street_name,project_name,price,area,frontage_width,bedroom_count,bathroom_count,house_direction,full_date,post_hour,is_outlier
0,Nhà trọ 1 trệt 1 lầu 10x28m mặt tiền kinh doan...,"Nhà mặt tiền kinh doanh gần chợ Thông Dụng, ph...",Nhà,Bình Dương,Thuận An,An Phú,Chưa xác định,Không thuộc dự án,1.200000e+10,280.00,NaN,0,0,Chưa xác định,2025-11-12,12,0
1,"Bán gấp nhà riêng 3PN, 3WC, 76m2 tại ĐT 743, B...","Nhà riêng tại Cát Tường J - Home, ĐT 743, Phườ...",Biệt thự/Nhà liền kề,Bình Dương,Thuận An,Bình Chuẩn,Chưa xác định,Cát Tường J-Home,3.950000e+09,76.00,4.0,3,3,Chưa xác định,2025-12-06,5,0
2,Chủ bán nhà nghỉ gần trung tâm hành chính Dĩ A...,"Chủ bán nhà nghỉ Linh Chi, gần trung tâm hành ...",Nhà,Bình Dương,Dĩ An,Dĩ An,Nguyễn An Ninh,Không thuộc dự án,1.500000e+10,127.20,5.0,18,18,Chưa xác định,2025-10-13,9,1
3,Bán gấp lô đất kế góc ngay khu dân cư cao cấp ...,Bán gấp lô đất ngay KDC cao cấp toàn nhà lớn. ...,Đất,Bình Dương,Thuận An,An Phú,Chưa xác định,Không thuộc dự án,3.500000e+09,80.00,5.0,0,0,Nam,2025-12-15,10,0
4,Bán nhanh 300m2 đất mặt tiền ngay trung tâm xã...,Chính chủ kẹt vốn cần bán gấp lô đất thổ cư mặ...,Đất,Bình Dương,Phú Giáo,Phước Sang,DH508,Không thuộc dự án,4.800000e+08,300.00,NaN,0,0,Chưa xác định,2025-10-10,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36298,Đất MT đường Bế Văn Đàn Bình Nhâm 1. Đường xe ...,Đất mặt tiền Bình Nhâm 01 đường nhựa thông ra ...,Đất,Bình Dương,Thuận An,Bình Nhâm,Bình Nhâm 01,Không thuộc dự án,8.800000e+09,218.00,9.9,0,0,Đông Nam,2025-08-22,23,0
36299,Bán đất mặt tiền ĐX 078 Định Hoà giá rẻ đất th...,Bán đất mặt tiền ĐX 078 Định Hoà giá rẻ. Đất t...,Đất,Bình Dương,Thủ Dầu Một,Định Hòa,DX 078,Không thuộc dự án,6.250000e+09,783.75,16.5,0,0,Chưa xác định,2025-08-29,7,0
36300,"Căn hộ 2PN - 2wc ngay Linh Xuân, liền kề Go 1,...","Bán căn hộ 2 phòng ngủ 2wc, Bcons Green View, ...",Căn hộ chung cư,Bình Dương,Dĩ An,Đông Hòa,Quốc lộ 1A,Bcons Green View,1.900000e+09,53.00,NaN,2,2,Chưa xác định,2025-03-09,15,0
36301,Căn 75m2 2PN The Gió Riverside giá giai đoạn 1...,Căn 75m2 2PN The Gió Riverside giá giai đoạn 1...,Căn hộ chung cư,Bình Dương,Dĩ An,Bình Thắng,Vành Đai 3,The Gió Riverside,2.850000e+09,75.00,NaN,2,2,Đông,2025-06-27,8,0
